# Transformers From Scratch

Building a complete transformer architecture with tokenization, embeddings, encoder, and decoder.

## Setup and Installation

In [ ]:
!pip install tokenizers

## Imports

In [ ]:
import tempfile
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

## TOKENIZER

Create and train a Byte Pair Encoding (BPE) tokenizer on sample text data.

In [ ]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

In [ ]:
text_data  = """Humpty Dumpty sat on a wall,
               Humpty Dumpty had a great fall;
               All the king's horses and all
               the king's men
              Couldn't put Humpty together
              again."""

temp_file = tempfile.NamedTemporaryFile(
    mode='w',
    delete=False,
    encoding='utf-8'
)

temp_file.write(text_data)
temp_file.close()

tokenizer.train(
    [temp_file.name],
    trainer
)

os.remove(temp_file.name)

In [ ]:
output = tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.tokens)

In [ ]:
Vocab_map = tokenizer.get_vocab()
print(Vocab_map)

In [ ]:
token_ids = torch.tensor([output.ids])

print("\nTOKEN IDS TENSOR:")
print(token_ids)

## TOKEN EMBEDDINGS

Convert token IDs to dense embeddings.

In [ ]:
vocabsize = tokenizer.get_vocab_size()
print(vocabsize)

embedding_dim = 8

embedding_layers = nn.Embedding(vocabsize, embedding_dim)
token_embeddings = embedding_layers(token_ids)

print("\nTOKEN EMBEDDINGS:")
print(token_embeddings)

## POSITIONAL ENCODING

Add positional information to embeddings using sine and cosine functions.

In [ ]:
seq_len = token_ids.shape[1]
positional_encoding = torch.zeros(seq_len, embedding_dim)
positions = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

div_term = torch.exp(torch.arange(0, embedding_dim, 2).float()
 * -(torch.log(torch.tensor(10000.0)) / embedding_dim))

positional_encoding[:, 0::2] = torch.sin(positions * div_term)
positional_encoding[:, 1::2] = torch.cos(positions * div_term)

print("\nPOSITIONAL ENCODING:")
print(positional_encoding)

## ADDING POSITIONAL ENCODING TO TOKEN EMBEDDINGS

Combine token embeddings with positional encodings.

In [ ]:
final_input_embeddings = token_embeddings + positional_encoding

print("\nFINAL INPUT EMBEDDINGS:")
print(final_input_embeddings)

## TRANSFORMER ENCODER

Implementing Multi-Head Attention and Transformer Encoder Block.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim

        self.values = nn.Linear(embed_dim, embed_dim, bias=False)
        self.keys = nn.Linear(embed_dim, embed_dim, bias=False)
        self.queries = nn.Linear(embed_dim, embed_dim, bias=False)
        self.fc_out = nn.Linear(embed_dim, embed_dim)

    def forward(self, values, keys, query, mask=None):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        V = self.values(values).view(N, value_len, self.num_heads, self.head_dim)
        K = self.keys(keys).view(N, key_len, self.num_heads, self.head_dim)
        Q = self.queries(query).view(N, query_len, self.num_heads, self.head_dim)

        energy = torch.einsum("nqhd,nkhd->nhqk", [Q, K])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.head_dim ** 0.5), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, V]).reshape(N, query_len, self.embed_dim)
        return self.fc_out(out)

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, forward_expansion, dropout):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, forward_expansion * embed_dim),
            nn.ReLU(),
            nn.Linear(forward_expansion * embed_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, value, key, query, mask=None):
        attention = self.attention(value, key, query, mask)
        x = self.norm1(self.dropout(attention) + query)
        forward = self.feed_forward(x)
        out = self.norm2(self.dropout(forward) + x)
        return out

## TRANSFORMER DECODER

Implementing Decoder Block with masked self-attention and encoder-decoder attention.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, forward_expansion, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(embed_dim)
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.transformer_block = TransformerBlock(embed_dim, num_heads, forward_expansion, dropout)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, trg_mask):
        attention = self.attention(x, x, x, trg_mask)
        query = self.dropout(self.norm(attention + x))
        out = self.transformer_block(value, key, query, src_mask)
        return out

## PUTTING IT ALL TOGETHER

Running the complete transformer architecture end-to-end.

In [ ]:
forward_expansion = 4
dropout = 0.1
num_heads = 2

encoder_block = TransformerBlock(embedding_dim, num_heads, forward_expansion, dropout)
decoder_block = DecoderBlock(embedding_dim, num_heads, forward_expansion, dropout)

In [ ]:
enc_out = encoder_block(final_input_embeddings, final_input_embeddings, final_input_embeddings)
dec_out = decoder_block(final_input_embeddings, enc_out, enc_out, src_mask=None, trg_mask=None)

In [ ]:
fc_out = nn.Linear(embedding_dim, vocabsize)
logits = fc_out(dec_out)
probabilities = torch.softmax(logits, dim=-1)
predicted_token = torch.argmax(probabilities, dim=-1)

print("Predicted tokens from full architecture:", predicted_token)

## Architecture Diagram

```
+----------------+
|  Raw Text Data |
+----------------+
         |
         v
+----------------+
|   Tokenizer    |
+----------------+
         |
         v
+----------------+
| Vocabulary Map |
+----------------+
         |
         v
+----------------------+
| Token ID Sequences   |
+----------------------+
         |
         v
+----------------------+
| Positional Encoding  |
+----------------------+
         |
         v
+----------------------+
| Token Embeddings     |
+----------------------+
         |
         v
+===========================================+
|          TRANSFORMER ENCODER              |
+===========================================+

   +-----------------------------------+
   | Multi-Head Self Attention         |
   |  - Query                          |
   |  - Key                            |
   |  - Value                          |
   +-----------------------------------+
                  |
                  v
   +-----------------------------------+
   | Add & Layer Normalization         |
   +-----------------------------------+
                  |
                  v
   +-----------------------------------+
   | Feed Forward Neural Network       |
   +-----------------------------------+
                  |
                  v
   +-----------------------------------+
   | Add & Layer Normalization         |
   +-----------------------------------+

         (Repeated N Times)

                  |
                  v

+===========================================+
|          TRANSFORMER DECODER              |
+===========================================+

   +-----------------------------------+
   | Masked Multi-Head Attention       |
   +-----------------------------------+
                  |
                  v
   +-----------------------------------+
   | Encoder-Decoder Attention         |
   +-----------------------------------+
                  |
                  v
   +-----------------------------------+
   | Feed Forward Network              |
   +-----------------------------------+

         (Repeated N Times)

                  |
                  v

+----------------------+
| Linear Projection    |
+----------------------+
         |
         v
+----------------------+
| Softmax Probabilities|
+----------------------+
         |
         v
+----------------------+
| Predicted Token      |
+----------------------+
```